# سبق 03 - ایجنٹک ڈیزائن پیٹرنز

اس سبق میں، ہم مؤثر AI ایجنٹس بنانے کے لیے تین بنیادی ڈیزائن پیٹرنز کا جائزہ لیتے ہیں:

1. **واضح ایجنٹ ہدایات** — ایسے درست، کردار کی تعریف کرنے والے پرامپس تیار کرنا جو ایجنٹ کے رویے کی رہنمائی کریں  
2. **Pydantic ماڈلز کے ساتھ منظم آؤٹ پٹ** — یہ یقینی بنانا کہ ایجنٹس پیش گوئی شدہ، تصدیق شدہ ڈیٹا واپس کریں  
3. **سنگل ریسپانسبلٹی ایجنٹس** — ایسے مرکوز ایجنٹس ڈیزائن کرنا جو ہر ایک چیز کو اچھی طرح انجام دیں  

ہم ہر پیٹرن کو ایک **ٹریول ڈیسٹینیشن ریکمنڈر** منظرنامے پر لاگو کریں گے، اور بتدریج ایک ایسا نظام بنائیں گے جو مقامات تجویز کر سکے، دستیابی چیک کر سکے، اور لاجسٹکس کو سنبھال سکے۔


## ترتیب


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## پیٹرن 1: ایجنٹ کے لیے واضح ہدایات

سب سے زیادہ مؤثر پیٹرن سب سے آسان بھی ہے: اپنے ایجنٹ کے لیے واضح، تفصیلی ہدایات لکھنا۔

اچھی ہدایات تعریف کرتی ہیں:
- **کون** ایجنٹ ہے (شخصیت اور لہجہ)
- **کیا** اسے کرنا چاہیے (قدم بہ قدم ذمہ داریاں)
- **کیسے** اسے برتاؤ کرنا چاہیے (حدود اور انداز)

نیچے، ہم ایک سفر کے کنسیئر ایجنٹ بناتے ہیں جس کے واضح ہدایات ہر جواب کو شکل دیتی ہیں جو وہ مہیا کرتا ہے۔


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## پیٹرن 2: پائیڈینٹک ماڈلز کے ساتھ ساختہ آؤٹ پٹ

مفت الفاظ میں متن گفتگو کے لیے مفید ہے، لیکن نیچے والے نظاموں کو ساختہ ڈیٹا کی ضرورت ہوتی ہے۔
**پائیڈینٹک ماڈلز** کو **ٹول فنکشن** کے ساتھ جوڑ کر، ہم:

- ایجنٹ کی آؤٹ پٹ کے لیے ایک مکمل اسکیمہ متعین کر سکتے ہیں
- جوابات کو خودکار طریقے سے تصدیق کر سکتے ہیں
- ایجنٹ کے نتائج کو قابل اعتماد طریقے سے ایپلیکیشن لاجک میں ضم کر سکتے ہیں

ہم ایک ایسا ٹول بھی متعارف کرواتے ہیں جو مقام کی تفصیلات واپس کرتا ہے تاکہ ایجنٹ اپنی سفارشات کو حقیقی ڈیٹا پر مبنی بنا سکے۔


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## پیٹرن 3: واحد ذمہ داری والے ایجنٹس

پیچیدہ کاموں سے فائدہ اٹھانے کے لیے کام کو کئی مخصوص ایجنٹس میں تقسیم کرنا مفید ہوتا ہے، جن میں سے ہر ایک کی ایک واحد ذمہ داری ہوتی ہے:

- ایک **مقصد کا ماہر** جو مقامات اور دستیابی کے بارے میں جانتا ہے
- ایک **لاجسٹکس پلانر** جو پروازوں، ہوٹلوں، اور سفرناموں کا انتظام کرتا ہے

یہ سافٹ ویئر انجینئرنگ کے اصول *علاقہ بندی کے اصول* کی عکاسی کرتا ہے — ہر ایجنٹ کو الگ سے ٹیسٹ کرنا، برقرار رکھنا، اور بہتر بنانا آسان ہوتا ہے۔


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## خلاصہ

اس سبق میں ہم نے سفر کی سفارش کرنے والے منظرنامے پر تین ایجنٹک ڈیزائن پیٹرنز لاگو کیے:

| پیٹرن | کلیدی خیال | فائدہ |
|---|---|---|
| **صاف ہدایات** | پہلے سے شخص، ذمہ داریاں، اور پابندیاں متعین کریں | مستقل، برانڈ کے مطابق ایجنٹ کا رویہ |
| **منظم آؤٹ پٹ** | جواب کے فارمیٹ کے طور پر پائیڈانٹک ماڈلز استعمال کریں | تصدیق شدہ، مشین پڑھنے کے قابل نتائج |
| **واحد ذمہ داری** | ہر ایجنٹ کو ایک منفرد اور مرکوز کام دیں | جانچنا، برقرار رکھنا، اور ترتیب دینا آسان |

یہ پیٹرنز فطری طور پر ایک ساتھ مل کر کام کرتے ہیں — آپ صاف ہدایات کو منظم آؤٹ پٹ کے ساتھ ایک واحد ذمہ داری والے ایجنٹ کے اندر جوڑ سکتے ہیں تاکہ مضبوط، پروڈکشن کے لیے تیار نظام بنایا جا سکے۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**معذرت**:  
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ اگرچہ ہم درستگی کی کوشش کرتے ہیں، براہ کرم آگاہ رہیں کہ خودکار تراجم میں غلطیاں یا عدم مطابقت ہو سکتی ہے۔ اصل دستاویز اپنی مادری زبان میں ماخذ کے طور پر معتبر سمجھی جانی چاہیے۔ اہم معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ ہم اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کے ذمہ دار نہیں ہیں۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
